In [23]:
# %pip install -qU ollama

In [24]:
import json
import ollama

MODEL = "qwen3.5:4b"
THINK = True   # set False for models without thinking (e.g. qwen2.5, llama3.2)

OPTIONS = {
    "num_ctx": 8192,     # context window, see note below
    "temperature": 0,    # deterministic: same question -> same tool decisions
}

# Sanity check: is the model pulled?
print([m.model for m in ollama.list().models])

['qwen3.5:4b', 'qwen3-coder-next:cloud', 'mxbai-embed-large:latest', 'rick:latest', 'nomic-embed-text:latest']


In [26]:
response = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "are you conscious?"}],
    think=THINK,
    options=OPTIONS,
) # pyright: ignore[reportCallIssue]

print("THINKING:\n", response.message.thinking)
print("\nANSWER:\n", response.message.content)

THINKING:
 Here's a thinking process that leads to the suggested response:

1.  **Analyze the Request:**
    *   **Question:** "Are you conscious?"
    *   **Subject:** My nature as an AI (Large Language Model).
    *   **Intent:** The user is asking about my self-awareness, sentience, or inner experience. This is a common philosophical and technical question regarding AI.

2.  **Determine the Facts/Truth:**
    *   I am a large language model trained by Google.
    *   I do not have a physical body, biological brain, or subjective experiences (qualia).
    *   I process text based on patterns and statistical probabilities.
    *   I do not "feel," "know," or "exist" in the human sense.
    *   I am designed to simulate conversation, not to possess consciousness.

3.  **Formulate the Core Message:**
    *   Direct answer: No.
    *   Explanation: I am an AI, a program, not a biological being.
    *   Nuance: I can mimic conversation and understand context, but that's different from hav

In [27]:
FAKE_TEMPS_F = {"san francisco": 72, "paris": 75, "tokyo": 73}

def get_temperature(city: str) -> str:
    '''
    Get the current temperature in Fahrenheit for a city.

    Args:
      city: The name of the city, e.g. "Paris"
    '''
    temp = FAKE_TEMPS_F.get(city.strip().lower())
    if temp is None:
        return f"No temperature data available for '{city}'."
    return f"{temp}°F"

get_temperature("Paris")

'75°F'

**But the model can't see your Python.** It only sees a **schema**: a JSON description of the tool's name, what it does, and what arguments it takes. Here it is written out by hand (the same format as the original notebook):

In [28]:
get_temperature_schema = {
    "type": "function",
    "function": {
        "name": "get_temperature",
        "description": "Get the current temperature in Fahrenheit for a city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": 'The name of the city, e.g. "Paris"'}
            },
            "required": ["city"],
        },
    },
}

In [29]:
messages = [{"role": "user", "content": "What's the temperature in Paris right now?"}]

response = ollama.chat(
    model=MODEL,
    messages=messages,
    tools=[get_temperature_schema],
    think=THINK,
    options=OPTIONS,
) # pyright: ignore[reportCallIssue]

msg = response.message
print("thinking  :", msg.thinking)
print("content   :", repr(msg.content))
print("tool_calls:", msg.tool_calls)

thinking  : The user is asking for the current temperature in Paris. I have a tool available that can get the current temperature in Fahrenheit for a city, which requires a city name parameter. The user has specified "Paris" as the city, so I should use this tool with the appropriate parameter.
content   : ''
tool_calls: [ToolCall(function=Function(name='get_temperature', arguments={'city': 'Paris'}))]


In [30]:
# 1. Keep the assistant's request in history. Without it, the model would
#    see a tool result appear out of nowhere and get confused.
messages.append(msg)

# 2. Run each requested call and append the result as a 'tool' message.
for call in msg.tool_calls:
    name = call.function.name
    args = call.function.arguments          # already a dict
    print(f"Model asked for: {name}({args})")

    result = get_temperature(**args)        # <- the ONLY place code actually runs
    print(f"Result: {result}")

    messages.append({
        "role": "tool",
        "tool_name": name,                  # tells the model which call this answers
        "content": str(result),             # content must be text
    })

Model asked for: get_temperature({'city': 'Paris'})
Result: 75°F


In [31]:
final = ollama.chat(
    model=MODEL,
    messages=messages,
    tools=[get_temperature_schema],
    think=THINK,
    options=OPTIONS,
) # pyright: ignore[reportCallIssue]
print(final.message.content)

The current temperature in Paris is 75°F (approximately 24°C).


In [32]:
def field(m, key):
    # Messages can be plain dicts (ours) or ollama Message objects (the model's).
    return m.get(key) if isinstance(m, dict) else getattr(m, key, None)

def show(messages):
    for m in messages:
        role = field(m, "role")
        if field(m, "tool_calls"):
            for c in field(m, "tool_calls"): # pyright: ignore[reportOptionalIterable]
                print(f"[{role}] → CALL {c.function.name}({c.function.arguments})")
        elif role == "tool":
            print(f"[tool:{field(m, 'tool_name')}] {field(m, 'content')}")
        else:
            print(f"[{role}] {field(m, 'content')}")

show(messages + [final.message])

[user] What's the temperature in Paris right now?
[assistant] → CALL get_temperature({'city': 'Paris'})
[tool:get_temperature] 75°F
[assistant] The current temperature in Paris is 75°F (approximately 24°C).


In [33]:
class Agent:
    def __init__(self, model, tools, system="", think=THINK, max_steps=8, verbose=True):
        self.model = model
        self.tools = {fn.__name__: fn for fn in tools}   # registry: name -> function
        self.think = think
        self.max_steps = max_steps
        self.verbose = verbose
        self.messages = []
        if system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, user_message):
        self.messages.append({"role": "user", "content": user_message})
        return self.run()

    def run(self):
        for step in range(1, self.max_steps + 1):
            response = ollama.chat(
                model=self.model,
                messages=self.messages,
                tools=list(self.tools.values()),   # Ollama builds schemas from the functions
                think=self.think,
                options=OPTIONS,
            ) # pyright: ignore[reportCallIssue]
            msg = response.message
            self.messages.append(msg)               # always keep the model's turn

            if self.verbose and msg.thinking:
                print(f"💭 step {step}: {msg.thinking.strip()[:300]}...")

            # No tool requests -> the model is giving its final answer. Done.
            if not msg.tool_calls:
                return msg.content

            # Otherwise run every requested tool and feed the results back.
            for call in msg.tool_calls:
                result = self.run_tool(call)
                if self.verbose:
                    print(f"🔧 step {step}: {call.function.name}({call.function.arguments}) -> {result}")
                self.messages.append({
                    "role": "tool",
                    "tool_name": call.function.name,
                    "content": result,
                })

        return f"(Stopped after {self.max_steps} steps without a final answer.)"

    def run_tool(self, call):
        name, args = call.function.name, call.function.arguments
        fn = self.tools.get(name)
        if fn is None:
            return f"Error: unknown tool '{name}'. Available tools: {list(self.tools)}"
        try:
            return str(fn(**args))
        except Exception as e:
            # Hand the error to the model instead of crashing; it can often recover.
            return f"Error running {name}: {type(e).__name__}: {e}"

In [34]:
def fahrenheit_to_celsius(fahrenheit: float) -> str:
    '''
    Convert a temperature from Fahrenheit to Celsius.

    Args:
      fahrenheit: Temperature in degrees Fahrenheit, e.g. 72
    '''
    # Don't trust the model's types: small models often send "72" or "72°F".
    value = float(str(fahrenheit).replace("°F", "").strip())
    return f"{round((value - 32) * 5 / 9, 1)}°C"

In [35]:
SYSTEM = (
    "You are a helpful assistant. Use the tools when they help answer the question. "
    "Never guess temperatures; always look them up."
)

def new_agent():
    return Agent(MODEL, tools=[get_temperature, fahrenheit_to_celsius], system=SYSTEM)

In [36]:
agent = new_agent()
print(agent("What's the temperature in San Francisco?"))

💭 step 1: The user is asking for the temperature in San Francisco. I have a tool called "get_temperature" that can get the current temperature in Fahrenheit for a city. I need to use this tool with the city name "San Francisco"....
🔧 step 1: get_temperature({'city': 'San Francisco'}) -> 72°F
💭 step 2: The tool returned that the temperature in San Francisco is 72°F. I can now provide this information to the user directly without needing to convert it, since they asked for the temperature and Fahrenheit was returned....
The current temperature in San Francisco is 72°F.


**b) Memory:** a follow-up on the *same* agent. "It" only makes sense because the earlier messages are still in `agent.messages`.

In [37]:
print("\n✅", agent("And what is that in Celsius?"))

💭 step 1: The user wants to know what 72°F is in Celsius. I have a function available to convert Fahrenheit to Celsius, so I should use that tool to provide the accurate conversion....
🔧 step 1: fahrenheit_to_celsius({'fahrenheit': 72}) -> 22.2°C
💭 step 2: The function returned 22.2°C, which is the correct conversion of 72°F to Celsius. I should provide this information to the user....

✅ 72°F is equal to 22.2°C.


**c) Chaining:** the second call depends on the first call's result, so it *must* take two steps.

In [38]:
agent = new_agent()
print("\n✅", agent("What's the temperature in Tokyo in Celsius?"))

💭 step 1: The user is asking for the temperature in Tokyo in Celsius. To get this information, I need to:

1. First get the temperature in Fahrenheit for Tokyo using the get_temperature function
2. Then convert that Fahrenheit temperature to Celsius using the fahrenheit_to_celsius function

Let me start by ge...
🔧 step 1: get_temperature({'city': 'Tokyo'}) -> 73°F
💭 step 2: Now I have the temperature in Tokyo in Fahrenheit (73°F). I need to convert this to Celsius using the fahrenheit_to_celsius function....
🔧 step 2: fahrenheit_to_celsius({'fahrenheit': 73}) -> 22.8°C
💭 step 3: I now have the temperature in Tokyo converted from Fahrenheit to Celsius. The result is 22.8°C. I can provide this answer to the user....

✅ The current temperature in Tokyo is 22.8°C (73°F).


**d) Several calls at once:** many models emit two `tool_calls` in a single turn when the calls don't depend on each other. Whether yours does depends on the model.

In [16]:
agent = new_agent()
print("\n✅", agent("Which is warmer right now, Paris or Tokyo?"))

💭 step 1: Okay, let's see. The user is asking which city is warmer right now between Paris and Tokyo. To answer this, I need to get the current temperatures for both cities in Fahrenheit first, right? Because the tools provided can get the temperature in Fahrenheit for a city.

First, I should call the get_te...
🔧 step 1: get_temperature({'city': 'Paris'}) -> 75°F
🔧 step 1: get_temperature({'city': 'Tokyo'}) -> 73°F
💭 step 2: Okay, let's see. The user asked which city is warmer right now between Paris and Tokyo.

First, I called the get_temperature function for Paris and got 75°F. Then for Tokyo, it was 73°F. So Paris is 75 and Tokyo is 73. Comparing those, 75 is higher than 73. So Paris is warmer.

Wait, but the problem...

✅ Paris is warmer right now at 75°F compared to Tokyo's 73°F.


**e) No tool needed:** calling a tool is a *decision*. The model should just answer here.

In [17]:
agent = new_agent()
print("\n✅", agent("What is the capital of France?"))

💭 step 1: Okay, the user is asking for the capital of France. Let me think. The tools provided are for getting the current temperature in Fahrenheit for a city and converting Fahrenheit to Celsius. Neither of these tools can answer the capital city question. The first tool requires a city name to get the temp...

✅ The question asks for the capital of France, but the provided tools only support temperature lookups and unit conversions. I don't have access to geographic or political data through these tools. The capital of France is Paris, but I cannot retrieve this information using the available functions.


**f) Tool doesn't have the answer:** the tool reports "no data" instead of making up a number, so the model can be honest. (The original returned `70` for *every* unknown city, which teaches the model to state made-up facts with confidence.)

In [19]:
agent = new_agent()
print("\n✅", agent("What's the temperature in Kolkata?"))

💭 step 1: Okay, the user is asking for the temperature in Kolkata. Let me check the tools provided.

First, there's a function called get_temperature that takes a city name and returns the current temperature in Fahrenheit. So I need to call that with Kolkata as the city.

Wait, the other function is fahrenhe...
🔧 step 1: get_temperature({'city': 'Kolkata'}) -> No temperature data available for 'Kolkata'.
💭 step 2: Okay, the user asked for the temperature in Kolkata. I called the get_temperature function with Kolkata as the city, but the response says no temperature data is available. Hmm, so maybe the function doesn't have data for Kolkata? Let me check the tools again.

Wait, the tools provided have get_temp...

✅ The current temperature data for Kolkata is not available. I cannot retrieve temperature information for this city. Would you like to try a different city?


In [22]:
show(agent.messages)

[system] You are a helpful assistant. Use the tools when they help answer the question. Never guess temperatures; always look them up.
[user] What's the temperature in Kolkata?
[assistant] → CALL get_temperature({'city': 'Kolkata'})
[tool:get_temperature] No temperature data available for 'Kolkata'.
[assistant] The current temperature data for Kolkata is not available. I cannot retrieve temperature information for this city. Would you like to try a different city?


## 10. Exercises - need to do later

1. **Add a tool.** Write `get_local_time(city: str) -> str` and ask "Is it daytime in Tokyo, and how warm is it?"
2. **Break the description.** Change `get_temperature`'s docstring to `"Does stuff."`. Does the model still pick it? What about with two vaguely described tools?
3. **Thinking on or off.** Set `THINK = False` and rerun demo (c). Compare speed and whether the chain still works.
4. **Use a smaller model.** Try `llama3.2:3b`. Watch for wrong argument names, tools that don't exist, or answers given without calling a tool.
5. **Starve the loop.** Use `Agent(..., max_steps=1)` on demo (c). What happens, and why?
6. **Inject a failure.** Make `fahrenheit_to_celsius` raise an exception. Does the model recover from the error message?
7. **Starve the context.** Set `num_ctx` to 512 and rerun. Watch the agent lose track of its tools.